# 03.2 — Custom Tools

- Plain functions, @tool, dict returns, Literal schema, async tools; passed via tools=[...].
- Run top-to-bottom; each section builds on the last.
- Run: python 03.2-custom_tools.py


### Episode brief

- Two ways to write a tool: a plain function, or the `@tool` decorator
- The docstring IS the tool description the model reads — write it for the model
- Custom tools are ADDITIVE: you keep every built-in tool as well
  Run: python 03-custom_tools.py


### Step 1: Imports and setup

- Key: `from deepagents import create_deep_agent`.
- Run this cell, then check the output before moving on.


In [1]:
# %% Step 1: Imports and setup
################################ Imports and Setup ################################
import asyncio
import os
from typing import Literal

from deepagents import create_deep_agent
from dotenv import load_dotenv
from langchain.tools import tool
from rich import print

# Load .env so the model name can be overridden without touching the code
load_dotenv()
MODEL = os.getenv("DEEPAGENTS_MODEL", "ollama:qwen3.5:2b")

### Step 2: A tool can be a plain Python function

- Key: `def word_count(text: str) -> int:`.
- Note: deepagents inspects the signature and docstring to build the tool schema. Keep parameter types simple (str/int/bool) — smaller models coerce them ...
- Run this cell, then check the output before moving on.


In [2]:
# %% Step 2: A tool can be a plain Python function
################################ Plain Function Tools ################################
# deepagents inspects the signature and docstring to build the tool schema.
# Keep parameter types simple (str/int/bool) — smaller models coerce them better.
def word_count(text: str) -> int:
    """Count how many words are in the given text.

    Args:
        text: The text whose words should be counted.
    """
    # Split on whitespace so punctuation stays attached to its word
    return len(text.split())

### Step 3: ...or use the @tool decorator

- Key: `def reverse_text(text: str) -> str:`.
- Note: Same result, but the decorator gives you explicit control (custom name, args schema, return_direct, etc.) and turns it into a real BaseTool object.
- Run this cell, then check the output before moving on.


In [3]:
# %% Step 3: ...or use the @tool decorator
################################ Decorator-Based Tools ################################
# Same result, but the decorator gives you explicit control (custom name,
# args schema, return_direct, etc.) and turns it into a real BaseTool object.
@tool
def reverse_text(text: str) -> str:
    """Reverse the characters of the given text.

    Args:
        text: The text to reverse.
    """
    # Slice with a step of -1 to walk the string backwards
    return text[::-1]

### Step 4: Return structured data for the model to reason over

- Key: `def lookup_product(name: str) -> dict:`.
- Run this cell, then check the output before moving on.


In [4]:
# %% Step 4: Return structured data for the model to reason over
################################ Structured Return Values ################################
# Keep a tiny in-memory catalog so the example needs no external service
PRODUCTS = {
    "widget": {"price": 9.99, "in_stock": True},
    "gadget": {"price": 24.50, "in_stock": False},
}


def lookup_product(name: str) -> dict:
    """Look up a product's price and stock status.

    Args:
        name: The product name, such as 'widget' or 'gadget'.
    """
    # Return an error object instead of raising so the model can recover
    return {"error": f"Unknown product: {name}"} if name not in PRODUCTS else PRODUCTS[name]

### Step 5: Use Literal to give the model a constrained input schema

- Key: `def convert_temperature(value: float, scale: Literal["C", "F"]) -> str:`.
- Run this cell, then check the output before moving on.


In [5]:
# %% Step 5: Use Literal to give the model a constrained input schema
################################ Constrained Input Schema ################################
@tool
def convert_temperature(value: float, scale: Literal["C", "F"]) -> str:
    """Convert a temperature between Celsius (C) and Fahrenheit (F).

    Args:
        value: The temperature to convert.
        scale: The scale of value. It must be C or F.
    """
    # Treat the value as Celsius and convert up to Fahrenheit
    if scale == "C":
        return f"{value:g} C is {value * 9 / 5 + 32:g} F"
    # Otherwise treat it as Fahrenheit and convert down to Celsius
    return f"{value:g} F is {(value - 32) * 5 / 9:g} C"

### Step 6: Async tools let the agent await an I/O-bound operation

- Key: `async def estimate_delivery(order_id: str) -> str:`.
- Run this cell, then check the output before moving on.


In [6]:
# %% Step 6: Async tools let the agent await an I/O-bound operation
################################ Async Tools ################################
@tool
async def estimate_delivery(order_id: str) -> str:
    """Estimate delivery for an order after checking its shipping service.

    Args:
        order_id: The customer order number.
    """
    # Await a short sleep so the tool exercises the event loop without a real service
    await asyncio.sleep(0.2)  # Simulates an API request without a real service.
    return f"Order {order_id} is scheduled to arrive in 2 business days."

### Step 7: Hand every custom tool to the agent

- Key: `create_deep_agent(tools, model, system_prompt)`.
- Note: `tools=` ADDS to the built-in suite — it never removes ls/read_file/task/etc.
- Run this cell, then check the output before moving on.


In [7]:
# %% Step 7: Hand every custom tool to the agent
################################ Agent Assembly ################################
# `tools=` ADDS to the built-in suite — it never removes ls/read_file/task/etc.
agent = create_deep_agent(
    model=MODEL,
    tools=[
        word_count,
        reverse_text,
        lookup_product,
        convert_temperature,
        estimate_delivery,
    ],
    system_prompt=(
        "You are a utility assistant. Use every named custom and built-in "
        "tool exactly as requested; do not use execute."
    ),
)

### Step 8: Each prompt combines a custom-tool pattern with built-in tools.

- Key: `see code`.
- Note: `execute` is omitted because the default StateBackend is not a shell backend.
- Run this cell, then check the output before moving on.


In [8]:
# %% Step 8: Each prompt combines a custom-tool pattern with built-in tools.
################################ Task Prompts ################################
# `execute` is omitted because the default StateBackend is not a shell backend.
# Pair each custom tool with a built-in so the trace exercises both paths
TASK_PROMPTS = [
    (
        "Use write_file to create /demo.txt with exactly 'deep agents are fun', then use "
        "read_file to read it. Use reverse_text and word_count on the read contents."
    ),
    (
        "Use write_file to create /catalog.txt containing 'widget gadget'. Use glob to "
        "find .txt files and grep to find 'gadget' in /catalog.txt. Then use "
        "lookup_product with the exact name 'gadget' and report its price and stock status."
    ),
    (
        "Use write_file to create /temperature.txt containing '68 F'. Read it, use "
        "convert_temperature to convert 68 F to Celsius, then use edit_file (not a "
        "second write_file) to replace exactly '68 F' with '20 C' and read it again."
    ),
    "Use ls to inspect the virtual filesystem root, then use estimate_delivery for order 12345.",
]

### Step 9: Inspect which tools the agent actually called

- Key: `async def main() -> None:`.
- Note: `.text` is provider-agnostic: it extracts the text from a message whether the provider returned a plain string or a list of content blocks.
- Notebook runs await main() directly; the .py script uses asyncio.run(main()).
- Run this cell, then check the output before moving on.


In [ ]:
################################ Execution Loop ################################
# Run agent against test tasks and trace tool execution
async def main() -> None:
    """Run every task prompt and print the tool-call trace plus final answer."""
    # Iterate through benchmark prompts sequentially
    for number, task in enumerate(TASK_PROMPTS, start=1):
        # Execute the agent workflow for the current prompt
        result = await agent.ainvoke({"messages": [{"role": "user", "content": task}]})
        print(f"\n[bold cyan]Task {number}:[/bold cyan] {task}")

        # Traverse message history to audit tool calls and execution results
        for message in result["messages"]:
            # Display any tool calls requested by the model
            for call in getattr(message, "tool_calls", []) or []:
                print(f"  [green]{call['name']}[/green]({call['args']})")

            # Output tool execution output when available
            if message.__class__.__name__ == "ToolMessage":
                print(f"  [yellow]{message.name}[/yellow] -> {message.text}")

        # Display the terminal response generated for the user
        print(f"  [bold green]Final answer:[/bold green] {result['messages'][-1].text}")


# Execute the asynchronous entry point within the existing event loop
await main()